# 00 - Whisper vs Faster-Whisper na 40 prirodnih srpskih snimaka


## Cilj eksperimenta

U ovom notebook-u vrši se osnovno poređenje originalnog Whisper modela i Faster-Whisper implementacije na skupu od 40 prirodnih audio zapisa na srpskom jeziku, snimljenih od strane tri govornika. Audio zapisi su podeljeni u četiri grupe prema dužini: kratki, srednji, dugi i veoma dugi.

Za poređenje se koristi isti Whisper `base` model u tri konfiguracije: originalni Whisper, Faster-Whisper sa INT8 kvantizacijom i Faster-Whisper sa FLOAT32 načinom izračunavanja. Analiziraju se vreme transkripcije, Real-Time Factor (RTF), Word Error Rate (WER) i potrošnja RAM memorije.

Cilj je da se utvrdi u kojoj meri optimizovana Faster-Whisper implementacija može da smanji vreme izvršavanja i memorijske zahteve, uz očuvanje kvaliteta transkripcije na srpskom jeziku.


In [ ]:
import os
import time
import wave
import gc
import re
from pathlib import Path

import psutil
import pandas as pd
import whisper
import jiwer
from faster_whisper import WhisperModel

# FFmpeg - putanja do ffmpeg.exe
os.environ["PATH"] += r";D:\anaconda\envs\whisper_test\Library\bin"

# Novi benchmark: 40 prirodnih srpskih snimaka / 3 govornika
AUDIO_FOLDER = Path(
    r"D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\audio\40Sentences00-08Notebooks"
)

# Rezultati ovog notebooka
PROJEKAT_FOLDER = AUDIO_FOLDER.parent.parent
REZULTATI_FOLDER = PROJEKAT_FOLDER / "rezultati" / "00_whisper_vs_faster_whisper"
REZULTATI_FOLDER.mkdir(parents=True, exist_ok=True)

def govornik_za_broj(i):
    if i in list(range(1, 5)) + list(range(11, 16)) + list(range(26, 29)) + [36]:
        return 1
    if i in list(range(5, 8)) + list(range(16, 21)) + list(range(29, 32)) + [37, 38]:
        return 2
    return 3

def grupa_duzine(i):
    if 1 <= i <= 10:
        return "kratki"
    if 11 <= i <= 25:
        return "srednji"
    if 26 <= i <= 35:
        return "dugi"
    return "veoma_dugi"

# Svih 40 novih prirodnih srpskih WAV fajlova
audio_fajlovi = [
    f"{i}govornik{govornik_za_broj(i)}.wav"
    for i in range(1, 41)
]

# Svi snimci su na srpskom
jezici_fajlova = ["sr"] * len(audio_fajlovi)

# Referentni tekstovi - moraju tačno odgovarati izgovorenim rečenicama
referentni_tekstovi_lista = ['Veštačka inteligencija omogućava računarima da rešavaju složene probleme i analiziraju velike količine podataka.', 'Automatsko prepoznavanje govora pretvara izgovorene reči u tekst koji računar može dalje da obrađuje.', 'Danas je veoma prijatan dan, pa smo odlučili da prošetamo kroz centar grada posle ručka.', 'Novi računari imaju brže procesore, više memorije i znatno bolje performanse nego prethodne generacije.', 'Mašinsko učenje koristi podatke kako bi model naučio da prepoznaje obrasce i donosi odgovarajuće odluke.', 'Sutra ujutru moram da završim nekoliko obaveza, a zatim planiram da se vidim sa prijateljima.', 'Kvalitet zvučnog zapisa može značajno da utiče na tačnost sistema za automatsko prepoznavanje govora.', 'Temperatura je tokom noći naglo pala, ali se očekuje da će sutra vreme ponovo biti toplije.', 'Neuronske mreže danas se koriste za obradu slike, teksta, zvuka i mnogih drugih vrsta podataka.', 'Dobar sistem mora istovremeno da bude dovoljno brz, precizan i pouzdan u različitim uslovima.', 'Savremeni sistemi za automatsko prepoznavanje govora koriste duboke neuronske mreže kako bi analizirali zvučni signal i pretvorili ga u tekst. Njihova tačnost zavisi od modela, kvaliteta snimka, jezika i karakteristika govornika.', 'Juče sam krenuo do grada nešto ranije nego obično. Na ulicama nije bilo mnogo ljudi, pa sam brzo završio sve obaveze. Nakon toga sam seo u obližnji kafić i sačekao prijatelja.', 'Whisper je model za automatsko prepoznavanje govora koji može da radi sa velikim brojem jezika. Faster-Whisper koristi optimizovanu implementaciju koja omogućava efikasnije izvršavanje i manju potrošnju računarskih resursa.', 'Kada razgovaramo u tihoj prostoriji, sistem uglavnom može veoma jasno da prepozna izgovorene reči. Međutim, saobraćaj, muzika, razgovor drugih ljudi i različiti izvori šuma mogu značajno da otežaju transkripciju.', 'Tokom razvoja softverske aplikacije programeri prvo analiziraju zahteve korisnika, zatim projektuju sistem i implementiraju potrebne funkcionalnosti. Nakon toga slede testiranje, ispravljanje grešaka i postavljanje aplikacije u produkciono okruženje.', 'Prošlog vikenda smo odlučili da provedemo dan u prirodi. Vreme je bilo sunčano i prijatno, pa smo poneli hranu i nekoliko stvari koje su nam bile potrebne. Vratili smo se kući tek predveče.', 'Veći modeli mašinskog učenja često mogu da postignu veću tačnost, ali zahtevaju više memorije i procesorske snage. Zbog toga najveći model nije uvek najbolji izbor za praktičnu primenu.', 'Brzina automatskog prepoznavanja govora posebno je važna kod aplikacija koje rade u realnom vremenu. Korisnik očekuje da sistem reaguje brzo i da se transkript pojavljuje bez primetnog kašnjenja.', 'Kada sam jutros uključio računar, prvo sam proverio elektronsku poštu i poruke. Posle toga sam otvorio projekat na kojem trenutno radim i nastavio implementaciju funkcionalnosti koju sam započeo prethodnog dana.', 'Srpski jezik sadrži glasove i karaktere koji se razlikuju od engleskog jezika. Zbog toga je interesantno analizirati koliko dobro isti model prepoznaje govor na različitim jezicima.', 'Kvantizacija omogućava da se neuronski model izvršava korišćenjem manje preciznih numeričkih formata. Na taj način moguće je smanjiti potrošnju memorije i ubrzati izvršavanje uz prihvatljivu promenu tačnosti.', 'Čovek tokom običnog razgovora često pravi kratke pauze, menja brzinu govora ili zastane kako bi razmislio. Sistem za prepoznavanje govora mora da bude dovoljno robustan da pravilno obradi takve situacije.', 'Danas postoji veliki broj aplikacija koje koriste veštačku inteligenciju. Ona se primenjuje u medicini, industriji, obrazovanju, saobraćaju, finansijama i mnogim svakodnevnim digitalnim servisima.', 'Kada se zvučnom signalu doda šum, neke reči postaju teže razumljive. Što je nivo šuma veći u odnosu na govor, sistemu je teže da napravi tačnu transkripciju celog snimka.', 'Pre nego što rezultate eksperimenta možemo pravilno da uporedimo, potrebno je koristiti iste ulazne podatke i jasno definisane metrike. Na taj način razlika u rezultatima može da se poveže sa promenom koju zapravo ispitujemo.', 'Automatsko prepoznavanje govora predstavlja oblast veštačke inteligencije koja se bavi pretvaranjem zvučnog signala u tekst. Savremeni sistemi koriste modele dubokog učenja koji mogu da analiziraju kompleksne karakteristike ljudskog govora. Njihove performanse mogu da zavise od jezika, kvaliteta mikrofona, prisustva pozadinskog šuma, brzine govora i karakteristika samog govornika. Zbog toga je prilikom evaluacije važno koristiti različite vrste zvučnih zapisa.', 'Jednog jutra krenuo sam automobilom prema gradu nešto ranije nego obično. Saobraćaj je u početku bio veoma slab, ali se približavanjem centru broj vozila postepeno povećavao. Dok sam čekao na semaforu, na radiju su govorili o vremenskoj prognozi za narednih nekoliko dana. Najavljeno je sunčano vreme, ali i mogućnost kratkotrajne kiše tokom vikenda. Kada sam stigao, parkirao sam automobil i nastavio pešice.', 'Whisper postoji u više veličina modela, od veoma malih do znatno većih modela. Manji modeli zahtevaju manje memorije i uglavnom omogućavaju bržu transkripciju, dok veći modeli mogu da pruže veću tačnost u zahtevnijim uslovima. Zbog toga izbor modela predstavlja kompromis između kvaliteta transkripcije i potrebnih računarskih resursa. Taj kompromis je posebno važan kada se sistem izvršava na običnom računaru bez snažne grafičke kartice.', 'Pozadinski šum predstavlja jedan od najčešćih problema kod sistema za automatsko prepoznavanje govora. U realnom okruženju mikrofon ne snima samo glas osobe već i zvuk automobila, ventilatora, drugih ljudi ili muzike. Ako je govor dovoljno snažan u odnosu na šum, transkripcija može ostati veoma dobra. Međutim, kada se odnos signala i šuma smanjuje, broj pogrešno prepoznatih reči uglavnom počinje da raste.', 'Kada razvijamo sistem koji treba da radi u realnom vremenu, nije dovoljno posmatrati samo njegovu tačnost. Potrebno je analizirati koliko vremena modelu treba da obradi određenu količinu zvuka. Ako obrada pet sekundi govora traje kraće od pet sekundi, sistem potencijalno može da prati govor bez stalnog povećavanja kašnjenja. Zbog toga su latencija i faktor realnog vremena veoma korisne metrike prilikom evaluacije ovakvih sistema.', 'Pre nekoliko dana organizovali smo malo okupljanje sa prijateljima. Dogovor je bio da se svi nađemo oko šest sati, ali je nekoliko ljudi kasnilo zbog saobraćaja. Dok smo ih čekali, razgovarali smo o poslu, putovanjima i planovima za naredne mesece. Kasnije smo naručili večeru i ostali zajedno nekoliko sati. Iako ništa posebno nije bilo planirano, veče je na kraju bilo veoma zanimljivo i prijatno.', 'Ljudski govor nije potpuno konstantan signal. Neko govori brzo, neko sporije, a pojedini govornici prave veoma duge pauze između rečenica. Razlikuju se i visina glasa, naglasak, način izgovora i jačina govora. Sistem koji postiže odlične rezultate kod jednog govornika zato ne mora nužno da postigne iste rezultate kod druge osobe. Korišćenje više govornika omogućava realniju procenu sposobnosti modela da generalizuje.', 'Voice Activity Detection koristi se za razlikovanje delova zvučnog zapisa u kojima postoji govor od delova u kojima se nalazi tišina. Ako snimak sadrži duge periode bez govora, njihovo uklanjanje može da smanji količinu podataka koju model mora da obradi. Međutim, algoritam mora pažljivo da odredi početak i kraj govora, jer previše agresivno uklanjanje može da odseče deo izgovorene reči i tako negativno utiče na transkripciju.', 'Beam search predstavlja jedan od načina pronalaženja verovatne sekvence reči tokom dekodiranja rezultata modela. Veća vrednost parametra omogućava razmatranje većeg broja mogućih kandidata, ali istovremeno može da poveća vreme potrebno za obradu. Zbog toga je korisno eksperimentalno proveriti da li povećanje beam size parametra zaista donosi značajno poboljšanje tačnosti i koliko dodatnih računarskih resursa zahteva.', 'Prilikom izvođenja eksperimenta veoma je važno menjati jednu karakteristiku sistema dok ostali uslovi ostaju isti. Ako istovremeno promenimo model, jezik, kvalitet zvuka i parametre dekodiranja, teško možemo da zaključimo zbog čega je rezultat postao bolji ili lošiji. Kontrolisani eksperimenti omogućavaju jasnije poređenje različitih konfiguracija i daju pouzdaniju osnovu za izvođenje zaključaka o ponašanju sistema.', 'Savremeni sistemi za automatsko prepoznavanje govora imaju veliki broj praktičnih primena. Mogu se koristiti za automatsko pravljenje transkripata sastanaka, generisanje titlova, glasovne asistente, pretragu multimedijalnog sadržaja i poboljšanje pristupačnosti digitalnih sistema. Ipak, kvalitet njihovog rada zavisi od velikog broja faktora. Govornici imaju različite glasove, akcente i brzine govora, dok snimci mogu nastati korišćenjem različitih mikrofona i u različitim prostorijama. Dodatni problem predstavlja pozadinski šum. Sistem koji veoma dobro radi u tihoj prostoriji ne mora da postigne isti rezultat na ulici ili u automobilu. Zbog toga evaluacija sistema ne bi trebalo da bude zasnovana na samo jednom zvučnom zapisu. Potrebno je koristiti više govornika, različite dužine snimaka i različite uslove. Pored tačnosti transkripcije, važno je analizirati vreme izvršavanja, potrošnju memorije i sposobnost sistema da obrađuje zvuk dovoljno brzo za praktičnu primenu.', 'Kada govorimo o performansama modela za prepoznavanje govora, postoje dve različite stvari koje želimo da postignemo. Prva je što tačniji transkript, a druga što efikasnije izvršavanje. Veći model može bolje da prepozna komplikovane rečenice ili govor u težim uslovima, ali istovremeno može zahtevati mnogo više memorije i vremena. Manji model može biti nešto manje precizan, ali dovoljno brz da se koristi na običnom laptopu. Kvantizacija predstavlja još jedan način optimizacije jer omogućava korišćenje numeričkog formata manje preciznosti. Time se može smanjiti količina potrebne memorije i ubrzati obrada. Međutim, svaka optimizacija mora da bude eksperimentalno proverena. Nije dovoljno pokazati da je jedna konfiguracija brža. Potrebno je proveriti i da li je njena tačnost ostala prihvatljiva. Upravo zato se koriste metrike kao što su Word Error Rate, latencija i Real-Time Factor, koje omogućavaju da različite konfiguracije modela budu upoređene na objektivan način.', 'Prošle nedelje imao sam prilično ispunjen dan. Ujutru sam ustao ranije jer sam morao da završim nekoliko obaveza pre odlaska u grad. Posle doručka proverio sam poruke, pripremio stvari koje su mi bile potrebne i krenuo automobilom. Na putu je bilo više saobraćaja nego što sam očekivao, pa mi je trebalo skoro pola sata da stignem do centra. Prvo sam otišao do prodavnice, zatim do banke, a nakon toga sam se našao sa prijateljem kojeg nisam video nekoliko meseci. Razgovarali smo o poslu, putovanjima i planovima za narednu godinu. Posle ručka vreme se iznenada promenilo i počela je jaka kiša. Sačekali smo nekoliko minuta da se vreme malo smiri, a onda smo krenuli kući. Kada sam se vratio, nastavio sam da radim na računaru. Uveče sam završio preostale obaveze i odlučio da ostatak dana provedem odmarajući se.', 'Sistem koji obrađuje govor u realnom vremenu mora da rešava nešto drugačiji problem od sistema koji dobije kompletan audio zapis. Kod klasične transkripcije model može da dobije čitav snimak i zatim ga obradi. Kod realnog vremena zvuk neprestano pristiže, pa ga je potrebno podeliti na manje delove. Ako koristimo veoma kratke segmente, sistem može brzo da prikaže rezultat, ali model ima manje konteksta za pravilno prepoznavanje rečenice. Sa druge strane, ako koristimo duge segmente, model dobija više konteksta, ali korisnik mora duže da čeka na rezultat. Zbog toga izbor veličine segmenta predstavlja kompromis između latencije i kvaliteta transkripcije. Na primer, mogu se uporediti segmenti od dve, pet i deset sekundi. Za svaku konfiguraciju moguće je izmeriti vreme obrade, faktor realnog vremena i tačnost konačnog transkripta. Takav eksperiment omogućava da se utvrdi koja konfiguracija predstavlja najbolji kompromis za praktičan sistem.', 'Cilj eksperimentalne analize sistema za automatsko prepoznavanje govora jeste da se utvrdi kako različiti faktori utiču na njegov rad. Možemo menjati veličinu modela, nivo pozadinskog šuma, jezik govornika, parametre dekodiranja, način kvantizacije ili korišćenje algoritma za detekciju govora. Svaki eksperiment treba organizovati tako da možemo jasno da utvrdimo posledice određene promene. Rezultati se zatim mogu posmatrati kroz više metrika. Word Error Rate pokazuje koliko se prepoznati tekst razlikuje od tačnog referentnog transkripta. Latencija pokazuje koliko vremena je potrebno za obradu, dok Real-Time Factor povezuje vreme obrade sa trajanjem samog zvučnog zapisa. Potrošnja memorije je posebno značajna kada model treba da radi na uređajima sa ograničenim resursima. Kombinovanjem ovih rezultata možemo da pronađemo konfiguraciju koja ne mora biti najbolja prema svakoj pojedinačnoj metrici, ali predstavlja dobar kompromis između tačnosti, brzine i potrebnih računarskih resursa.']

referentni_tekstovi = {
    audio_fajlovi[i]: referentni_tekstovi_lista[i]
    for i in range(40)
}

print("Podešavanje je završeno.")
print(f"Audio folder: {AUDIO_FOLDER}")
print(f"Broj audio fajlova: {len(audio_fajlovi)}")
print(f"Rezultati: {REZULTATI_FOLDER}")

In [ ]:
def trajanje_wav(putanja):
    """Vraća trajanje WAV fajla u sekundama."""
    with wave.open(str(putanja), "rb") as f:
        frames = f.getnframes()
        rate = f.getframerate()
    return frames / float(rate)


def ram_mb():
    """Vraća trenutno zauzeće RAM-a procesa u MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


def izmeri_vreme_i_ram(funkcija):
    """Meri vreme izvršavanja i promenu RAM memorije procesa."""
    gc.collect()
    ram_pre = ram_mb()
    pocetak = time.perf_counter()
    rezultat = funkcija()
    kraj = time.perf_counter()
    ram_posle = ram_mb()

    vreme = kraj - pocetak
    ram_promena = max(ram_posle - ram_pre, 0)
    return rezultat, vreme, ram_promena


def normalizuj_za_wer(tekst):
    """Lowercase + uklanjanje interpunkcije + sređivanje razmaka."""
    tekst = tekst.lower().strip()
    tekst = re.sub(r"[^\w\sčćžšđ]", " ", tekst, flags=re.UNICODE)
    tekst = re.sub(r"\s+", " ", tekst).strip()
    return tekst


def izracunaj_wer(referenca, hipoteza):
    return jiwer.wer(
        normalizuj_za_wer(referenca),
        normalizuj_za_wer(hipoteza)
    )

## Metodologija merenja

Za svaki audio zapis određuje se njegovo trajanje, nakon čega se ista ulazna datoteka transkribuje pomoću sve tri konfiguracije modela. Vreme izvršavanja meri se funkcijom `time.perf_counter()`, dok se promena zauzeća memorije procesa prati pomoću biblioteke `psutil`.

Kvalitet transkripcije procenjuje se pomoću metrike Word Error Rate (WER). Pre izračunavanja WER-a referentni i generisani tekst normalizuju se pretvaranjem u mala slova, uklanjanjem interpunkcije i sređivanjem razmaka. Na ovaj način poređenje je prvenstveno usmereno na greške u prepoznatim rečima, a ne na razlike u formatiranju teksta.

Pored vremena izvršavanja izračunava se i Real-Time Factor (RTF) kao odnos vremena obrade i trajanja audio zapisa. Vrednost RTF manja od 1 označava da se audio obrađuje brže od njegovog realnog trajanja.


In [ ]:
print("PROVERA AUDIO FAJLOVA")
print("-" * 100)

nedostaju = []

for i, (fajl, jezik) in enumerate(zip(audio_fajlovi, jezici_fajlova), start=1):
    putanja = AUDIO_FOLDER / fajl
    govornik = govornik_za_broj(i)
    grupa = grupa_duzine(i)

    if putanja.exists():
        trajanje = trajanje_wav(putanja)
        print(
            f"{i:02d} | {fajl:<18} | govornik {govornik} | "
            f"{grupa:<10} | {jezik} | POSTOJI | {trajanje:.2f} s"
        )
    else:
        nedostaju.append(fajl)
        print(
            f"{i:02d} | {fajl:<18} | govornik {govornik} | "
            f"{grupa:<10} | {jezik} | NE POSTOJI"
        )

print("-" * 100)
print(f"Pronađeno: {40 - len(nedostaju)}/40")

if nedostaju:
    print("Nedostaju fajlovi:")
    for fajl in nedostaju:
        print(" -", fajl)
else:
    print("Sva 40 audio fajla su pronađena.")

In [ ]:
print("Učitavanje Original Whisper modela...")

model_orig = whisper.load_model("base")

print("Original Whisper je uspešno učitan.")

In [ ]:
print("Učitavanje Faster-Whisper INT8 modela...")

model_fw_int8 = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

print("Faster-Whisper INT8 je uspešno učitan.")

In [ ]:
print("Učitavanje Faster-Whisper FLOAT32 modela...")

model_fw_float32 = WhisperModel(
    "base",
    device="cpu",
    compute_type="float32"
)

print("Faster-Whisper FLOAT32 je uspešno učitan.")

## Poređenje Whisper konfiguracija

Svih 40 audio zapisa obrađuje se korišćenjem tri konfiguracije: originalnog Whisper `base` modela, Faster-Whisper `base` modela sa INT8 kvantizacijom i Faster-Whisper `base` modela sa FLOAT32 načinom izračunavanja.

Za svaki pojedinačni snimak čuvaju se trajanje audio zapisa, vreme transkripcije, promena RAM memorije, RTF i WER, kao i kompletni generisani transkripti. Dodatno se beleže govornik i grupa dužine snimka, što omogućava kasniju analizu uticaja trajanja audio zapisa i karakteristika govornika na performanse sistema.

Korišćenjem istih audio zapisa i istog `base` modela omogućeno je kontrolisano poređenje implementacija, pri čemu su glavne razlike način izvršavanja modela i numerička preciznost.


In [ ]:
rezultati = []

for i, (fajl, jezik) in enumerate(zip(audio_fajlovi, jezici_fajlova), start=1):

    putanja = AUDIO_FOLDER / fajl

    if not putanja.exists():
        print(f"Fajl {fajl} ne postoji - preskačem.")
        continue

    print("\n" + "=" * 100)
    print(f"[{i:02d}/40] Obrada fajla: {fajl}")
    print(f"Govornik: {govornik_za_broj(i)} | Grupa: {grupa_duzine(i)} | Jezik: {jezik}")

    trajanje = trajanje_wav(putanja)
    ref_tekst = referentni_tekstovi[fajl]

    # 1. ORIGINAL WHISPER
    def run_original():
        rezultat = model_orig.transcribe(
            str(putanja),
            language=jezik,
            fp16=False
        )
        return rezultat["text"].strip()

    tekst_orig, vreme_orig, ram_orig = izmeri_vreme_i_ram(run_original)
    wer_orig = izracunaj_wer(ref_tekst, tekst_orig)

    # 2. FASTER-WHISPER INT8
    def run_faster_int8():
        segments, info = model_fw_int8.transcribe(
            str(putanja),
            language=jezik
        )
        return " ".join(
            segment.text.strip()
            for segment in segments
        ).strip()

    tekst_fw_int8, vreme_fw_int8, ram_fw_int8 = izmeri_vreme_i_ram(run_faster_int8)
    wer_fw_int8 = izracunaj_wer(ref_tekst, tekst_fw_int8)

    # 3. FASTER-WHISPER FLOAT32
    def run_faster_float32():
        segments, info = model_fw_float32.transcribe(
            str(putanja),
            language=jezik
        )
        return " ".join(
            segment.text.strip()
            for segment in segments
        ).strip()

    tekst_fw_float32, vreme_fw_float32, ram_fw_float32 = izmeri_vreme_i_ram(run_faster_float32)
    wer_fw_float32 = izracunaj_wer(ref_tekst, tekst_fw_float32)

    rezultati.append({
        "Broj": i,
        "Audio": fajl.replace(".wav", ""),
        "Govornik": f"govornik{govornik_za_broj(i)}",
        "Grupa_duzine": grupa_duzine(i),
        "Jezik": jezik,
        "Trajanje [s]": round(trajanje, 2),

        "Vreme_Orig [s]": round(vreme_orig, 2),
        "RAM_Orig [MB]": round(ram_orig, 2),
        "RTF_Orig": round(vreme_orig / trajanje, 4),
        "WER_Orig": round(wer_orig, 4),

        "Vreme_FW_Int8 [s]": round(vreme_fw_int8, 2),
        "RAM_FW_Int8 [MB]": round(ram_fw_int8, 2),
        "RTF_FW_Int8": round(vreme_fw_int8 / trajanje, 4),
        "WER_FW_Int8": round(wer_fw_int8, 4),

        "Vreme_FW_F32 [s]": round(vreme_fw_float32, 2),
        "RAM_FW_F32 [MB]": round(ram_fw_float32, 2),
        "RTF_FW_F32": round(vreme_fw_float32 / trajanje, 4),
        "WER_FW_F32": round(wer_fw_float32, 4),

        "Referenca": ref_tekst,
        "Transkript_Orig": tekst_orig,
        "Transkript_FW_Int8": tekst_fw_int8,
        "Transkript_FW_F32": tekst_fw_float32
    })

    print(f"Original Whisper:       {vreme_orig:.2f} s | WER {wer_orig*100:.2f}%")
    print(f"Faster-Whisper INT8:    {vreme_fw_int8:.2f} s | WER {wer_fw_int8*100:.2f}%")
    print(f"Faster-Whisper FLOAT32: {vreme_fw_float32:.2f} s | WER {wer_fw_float32*100:.2f}%")

In [ ]:
df = pd.DataFrame(rezultati)

print("SVA TESTIRANJA SU ZAVRŠENA.")

display(df)

## Eksperiment 2 - Analiza vremena izvršavanja

U ovom delu porede se vremena potrebna za transkripciju svih 40 audio zapisa. Za svaki snimak izračunava se faktor ubrzanja Faster-Whisper INT8 i FLOAT32 konfiguracije u odnosu na originalni Whisper.

Faktor ubrzanja definiše se kao odnos vremena izvršavanja originalnog Whisper modela i vremena izvršavanja odgovarajuće Faster-Whisper konfiguracije. Vrednost veća od 1 označava da je Faster-Whisper brži od originalne implementacije.

Pored pojedinačnih rezultata prikazuju se i prosečne vrednosti, čime se dobija pregled ukupnog ponašanja modela na kompletnom benchmark skupu.


In [ ]:
tabela1 = df[
    [
        "Audio",
        "Jezik",
        "Trajanje [s]",
        "Vreme_Orig [s]",
        "Vreme_FW_Int8 [s]",
        "Vreme_FW_F32 [s]"
    ]
].copy()

# Faktor ubrzanja INT8 u odnosu na Original Whisper
tabela1["Ubrzanje INT8"] = (
    tabela1["Vreme_Orig [s]"] /
    tabela1["Vreme_FW_Int8 [s]"]
).round(2)

# Faktor ubrzanja FLOAT32 u odnosu na Original Whisper
tabela1["Ubrzanje FLOAT32"] = (
    tabela1["Vreme_Orig [s]"] /
    tabela1["Vreme_FW_F32 [s]"]
).round(2)

display(tabela1)

In [ ]:
numericke_kolone = [
    "Trajanje [s]",
    "Vreme_Orig [s]",
    "Vreme_FW_Int8 [s]",
    "Vreme_FW_F32 [s]",
    "Ubrzanje INT8",
    "Ubrzanje FLOAT32"
]

prosek = tabela1[numericke_kolone].mean()

red_proseka = {
    "Audio": "PROSEK",
    "Jezik": "sr"
}

for kolona in numericke_kolone:
    red_proseka[kolona] = round(prosek[kolona], 2)

tabela1_sa_prosekom = pd.concat(
    [
        tabela1,
        pd.DataFrame([red_proseka])
    ],
    ignore_index=True
)

display(tabela1_sa_prosekom)

In [ ]:
tabela1_sa_prosekom.to_csv(
    REZULTATI_FOLDER / "tabela1_uporedna_analiza_vremena.csv",
    index=False,
    encoding="utf-8-sig"
)

df.to_csv(
    REZULTATI_FOLDER / "svi_rezultati_whisper.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tabela 1 i kompletni rezultati su uspešno sačuvani.")

## Eksperiment 3 - Poređenje performansi


In [ ]:
tabela3 = pd.DataFrame({
    "Audio": df["Audio"],
    "Jezik": df["Jezik"],
    "Trajanje [s]": df["Trajanje [s]"],
    "Whisper [s]": df["Vreme_Orig [s]"],
    "Faster-Whisper [s]": df["Vreme_FW_Int8 [s]"]
})

tabela3["Ubrzanje"] = (
    tabela3["Whisper [s]"] /
    tabela3["Faster-Whisper [s]"]
).round(2)

display(tabela3)

In [ ]:
prosecno_ukupno_ubrzanje = (
    tabela3["Whisper [s]"].sum() /
    tabela3["Faster-Whisper [s]"].sum()
)

prosecno_ubrzanje_po_zapisima = tabela3["Ubrzanje"].mean()

print(
    f"Ukupno ubrzanje Faster-Whisper INT8: "
    f"{prosecno_ukupno_ubrzanje:.2f}x"
)

print(
        f"Prosečan faktor ubrzanja po audio zapisima: "
    f"{prosecno_ubrzanje_po_zapisima:.2f}x"
)

In [ ]:
tabela3.to_csv(
    REZULTATI_FOLDER / "tabela3_poredjenje_whisper_faster.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tabela 3 je uspešno sačuvana.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

plt.plot(
    tabela3["Trajanje [s]"],
    tabela3["Whisper [s]"],
    marker="o",
    label="Original Whisper"
)

plt.plot(
    tabela3["Trajanje [s]"],
    tabela3["Faster-Whisper [s]"],
    marker="o",
    label="Faster-Whisper (INT8)"
)

plt.xlabel("Trajanje audio zapisa [s]")
plt.ylabel("Vreme transkripcije [s]")
plt.title("Poređenje vremena transkripcije: Whisper vs Faster-Whisper")
plt.legend()
plt.grid(True)

plt.savefig(
    REZULTATI_FOLDER / "grafik_poredjenje_whisper_faster.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Komentar rezultata

Poređenje vremena transkripcije omogućava direktnu procenu računarske efikasnosti originalnog Whisper modela i Faster-Whisper INT8 implementacije. Faktor ubrzanja veći od 1 označava prednost Faster-Whisper implementacije, dok ukupni faktor ubrzanja, izračunat na osnovu ukupnog vremena obrade svih snimaka, daje reprezentativniju procenu performansi kompletnog benchmark skupa.

Grafik prikazuje odnos između trajanja audio zapisa i vremena potrebnog za njegovu transkripciju. Očekuje se rast vremena obrade sa povećanjem trajanja snimka, dok razmak između dve krive pokazuje razliku u efikasnosti implementacija.

Originalni Whisper koristi PyTorch implementaciju, dok Faster-Whisper koristi CTranslate2 optimizovano izvršavanje. Zbog toga ovo poređenje pokazuje koliko promena implementacije i korišćenje INT8 konfiguracije mogu da utiču na brzinu izvršavanja istog `base` modela na CPU-u.

Rezultate vremena izvršavanja ne treba posmatrati izolovano, već zajedno sa WER i memorijskim rezultatima, jer praktično korisna optimizacija treba da obezbedi ubrzanje bez neprihvatljivog smanjenja kvaliteta transkripcije.


## Eksperiment 4 - RAM potrošnja


In [ ]:
import gc
import os
import psutil

proces = psutil.Process(os.getpid())

def ram_mb():
    return proces.memory_info().rss / (1024 * 1024)

print(f"Trenutna RAM potrošnja procesa: {ram_mb():.2f} MB")

In [ ]:
# Oslobađanje prethodno učitanih modela
try:
    del model_orig
except NameError:
    pass

try:
    del model_fw_int8
except NameError:
    pass

try:
    del model_fw_float32
except NameError:
    pass

gc.collect()

ram_pre_whisper = ram_mb()

model_whisper_ram = whisper.load_model("base")

ram_posle_whisper = ram_mb()

ram_whisper = ram_posle_whisper - ram_pre_whisper

print(f"RAM pre učitavanja:   {ram_pre_whisper:.2f} MB")
print(f"RAM posle učitavanja: {ram_posle_whisper:.2f} MB")
print(f"Promena RAM-a:        {ram_whisper:.2f} MB")

In [ ]:
# Oslobađanje Original Whisper modela
del model_whisper_ram
gc.collect()

ram_pre_faster = ram_mb()

model_faster_ram = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

ram_posle_faster = ram_mb()

ram_faster = ram_posle_faster - ram_pre_faster

print(f"RAM pre učitavanja:   {ram_pre_faster:.2f} MB")
print(f"RAM posle učitavanja: {ram_posle_faster:.2f} MB")
print(f"Promena RAM-a:        {ram_faster:.2f} MB")

In [ ]:
tabela4 = pd.DataFrame({
    "Model": [
        "Original Whisper",
        "Faster-Whisper INT8"
    ],
    "RAM [MB]": [
        round(ram_whisper, 2),
        round(ram_faster, 2)
    ]
})

display(tabela4)

In [ ]:
tabela4.to_csv(
    REZULTATI_FOLDER / "tabela4_ram_potrosnja.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tabela 4 je uspešno sačuvana.")

## Komentar rezultata - RAM potrošnja

U ovom eksperimentu poredi se dodatno zauzeće RAM memorije prilikom učitavanja originalnog Whisper `base` modela i Faster-Whisper `base` modela u INT8 konfiguraciji. Pre svakog merenja prethodno učitani model se uklanja iz memorije i poziva se garbage collector kako bi se smanjio uticaj prethodnog merenja.

Razlika između memorije procesa pre i posle učitavanja modela koristi se kao procena dodatne RAM memorije potrebne za njegovo učitavanje. INT8 reprezentacija koristi numerički format manje preciznosti, pa može omogućiti smanjenje memorijskih zahteva u odnosu na standardnu implementaciju.

Dobijene vrednosti predstavljaju potrošnju u konkretnom eksperimentalnom okruženju i ne treba ih tumačiti kao apsolutnu veličinu samog modela. Na rezultat mogu uticati Python runtime, prethodno učitane biblioteke, način upravljanja memorijom i operativni sistem. Zbog toga je najvažnije relativno poređenje konfiguracija izvršenih u istom okruženju.


## Eksperiment 5 - Detaljna analiza kvaliteta transkripcije na reprezentativnom snimku

In [ ]:
from jiwer import wer

# Jedan reprezentativan veoma dug prirodni srpski snimak
test_audio_naziv = "36govornik1.wav"
test_audio = AUDIO_FOLDER / test_audio_naziv
referenca = referentni_tekstovi[test_audio_naziv]

print("Test audio:")
print(test_audio)

print("\nReferentni tekst:")
print(referenca)

In [ ]:
model_whisper_wer = whisper.load_model("base")

rezultat_whisper = model_whisper_wer.transcribe(
    str(test_audio),
    language="sr",
    fp16=False
)

tekst_whisper = rezultat_whisper["text"].strip()

print("ORIGINAL WHISPER:")
print(tekst_whisper)

In [ ]:
model_faster_wer = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

segments, info = model_faster_wer.transcribe(
    str(test_audio),
    language="sr"
)

tekst_faster = " ".join(
    segment.text.strip()
    for segment in segments
).strip()

print("FASTER-WHISPER:")
print(tekst_faster)

In [ ]:
wer_whisper = izracunaj_wer(
    referenca,
    tekst_whisper
)

wer_faster = izracunaj_wer(
    referenca,
    tekst_faster
)

tabela5 = pd.DataFrame({
    "Model": [
        "Original Whisper",
        "Faster-Whisper INT8"
    ],
    "WER": [
        round(wer_whisper, 4),
        round(wer_faster, 4)
    ],
    "WER [%]": [
        round(wer_whisper * 100, 2),
        round(wer_faster * 100, 2)
    ]
})

display(tabela5)

In [ ]:
print("=" * 80)
print("REFERENTNI TEKST")
print("=" * 80)
print(referenca)

print("\n" + "=" * 80)
print("ORIGINAL WHISPER")
print("=" * 80)
print(tekst_whisper)

print("\n" + "=" * 80)
print("FASTER-WHISPER")
print("=" * 80)
print(tekst_faster)

In [ ]:
tabela5.to_csv(
    REZULTATI_FOLDER / "tabela5_wer.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tabela 5 je uspešno sačuvana.")

In [ ]:
if abs(wer_whisper - wer_faster) < 1e-9:
    print(
        "Zaključak: Oba modela imaju identičnu WER vrednost "
        "na testiranom audio zapisu."
    )
elif wer_faster < wer_whisper:
    print(
        "Zaključak: Faster-Whisper ostvaruje niži WER, "
        "odnosno bolji rezultat na testiranom audio zapisu."
    )
else:
    print(
        "Zaključak: Originalni Whisper ostvaruje niži WER "
        "na testiranom audio zapisu."
    )

## Komentar rezultata - kvalitet transkripcije

Kvalitet transkripcije poredi se pomoću metrike Word Error Rate (WER), koja uzima u obzir zamene, brisanja i umetanja reči u odnosu na referentni transkript. Niža WER vrednost označava precizniju transkripciju, dok WER jednak 0 predstavlja potpuno podudaranje sa referentnim tekstom nakon primenjene normalizacije.

Za detaljno poređenje izabran je veoma dug prirodni srpski audio zapis `36govornik1.wav`. Originalni Whisper i Faster-Whisper INT8 obrađuju isti snimak korišćenjem `base` modela, nakon čega se njihovi transkripti porede sa identičnim referentnim tekstom.

Ovaj pojedinačni primer omogućava detaljan pregled konkretnih razlika između generisanih transkripata, ali ga ne treba posmatrati kao procenu ukupne tačnosti modela. Reprezentativnija procena kvaliteta nalazi se u rezultatima obrade svih 40 audio zapisa, gde je WER izračunat za svaki snimak.

Zajedničkim posmatranjem WER-a, vremena izvršavanja, RTF-a i memorijske potrošnje moguće je proceniti da li Faster-Whisper INT8 pruža povoljniji kompromis između tačnosti i računarske efikasnosti u odnosu na originalni Whisper.
